# PCOS GNN — Notebook 3: Final Test Set Evaluation

**All five models trained on the full 80% training pool — evaluated once on sealed 20% test set**

---

**Models:** GCN · GAT · GraphSAGE · MPNN · GIN

**Protocol:** 150 fixed epochs per model · No early stopping · Same graph/Node2Vec/SMOTE for all

**Before running:**
1. Upload `pcos_test_set.csv` as Kaggle dataset named `pcos-test-set`
2. Attach both `pcos-cleaned-dataset` and `pcos-test-set` as inputs
3. Enable GPU accelerator
4. Run Cell 1 → restart kernel → run Cells 2–10 in order

**Important:** Do not re-run Cell 9 after viewing results. The test set is evaluated exactly once per model.

---

**Plots produced per model (7 per model = 35 total):**
Training loss curve · Node2Vec t-SNE · Confusion matrix · ROC + PR curves · DET curve · Lift + Gains · Precision-Threshold

**Comparison plots (4 total):**
ROC overlay · PR overlay · Metric comparison bar chart · Final ranking

## Cell 1 — Installation

In [19]:
# CELL 1 — INSTALLATION
# Run this cell first. Restart the kernel after it completes.

import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                   check=True)

pip_install('torch_geometric')
pip_install('imbalanced-learn')

print("Installation complete. Restart the kernel, then run Cell 2 onwards.")

Installation complete. Restart the kernel, then run Cell 2 onwards.


## Cell 2 — Imports, Seed & Configuration

In [20]:
# CELL 2 — IMPORTS, SEED, CONFIGURATION

import os, gc, random, warnings
warnings.filterwarnings('ignore')

import numpy  as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn.functional as F
from torch.nn import (Linear, Sequential, BatchNorm1d, ReLU)

import torch_geometric
from torch_geometric.data  import Data
from torch_geometric.nn    import (GCNConv, GATConv, SAGEConv,
                                    GINConv, NNConv)
from torch_geometric.utils import coalesce

from sklearn.model_selection  import train_test_split
from sklearn.preprocessing    import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold          import TSNE
from sklearn.metrics           import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, roc_auc_score, average_precision_score,
    cohen_kappa_score, confusion_matrix, fowlkes_mallows_score,
    normalized_mutual_info_score, roc_curve, precision_recall_curve,
    det_curve
)

from imblearn.over_sampling import SMOTE
from gensim.models          import Word2Vec

print(f"PyTorch           : {torch.__version__}")
print(f"PyTorch Geometric : {torch_geometric.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

GLOBAL_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(GLOBAL_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device            : {DEVICE}")

CONFIG = {
    'CLEANED_PATH' : ('/kaggle/input/datasets/monamehrun/'
                      'pcos-cleaned-dataset/pcos_cleaned.csv'),
    'TEST_PATH'    : '/kaggle/input/datasets/monamehrun/pcos-test-set/pcos_test_set.csv',
    'OUTPUT_DIR'   : '/kaggle/working/',
    'TARGET_COL'   : 'PCOS',
    'TEST_SIZE'    : 0.20,
    'SEED'         : GLOBAL_SEED,
    'K_NEIGHBOURS' : 10,
    'N2V_DIM'      : 64,
    'N2V_WALK_LEN' : 20,
    'N2V_CONTEXT'  : 10,
    'N2V_WALKS'    : 10,
    'N2V_EPOCHS'   : 50,
    'N2V_LR'       : 0.01,
    'HIDDEN_DIM'   : 64,
    'DROPOUT'      : 0.3,
    'FINAL_EPOCHS' : 150,
    'LR'           : 0.001,
    'WEIGHT_DECAY' : 1e-4,
}

MODELS_TO_RUN = ['GCN', 'GAT', 'GRAPHSAGE', 'MPNN', 'GIN']
print("\nModels to evaluate:", MODELS_TO_RUN)
print("Configuration loaded.")

PyTorch           : 2.10.0+cu128
PyTorch Geometric : 2.7.0
CUDA available    : True
Device            : cuda

Models to evaluate: ['GCN', 'GAT', 'GRAPHSAGE', 'MPNN', 'GIN']
Configuration loaded.


## Cell 3 — Load Data & Reconstruct Split

In [21]:
# CELL 3 — LOAD DATA AND RECONSTRUCT SPLIT

df_full = pd.read_csv(CONFIG['CLEANED_PATH'])
y_full  = df_full[CONFIG['TARGET_COL']].values.astype(np.int64)
X_full  = df_full.drop(columns=[CONFIG['TARGET_COL']]).values.astype(np.float32)
feature_names = df_full.drop(columns=[CONFIG['TARGET_COL']]).columns.tolist()

print(f"Full dataset: {X_full.shape[0]} rows x {X_full.shape[1]} features")

X_train, _, y_train, _ = train_test_split(
    X_full, y_full,
    test_size    = CONFIG['TEST_SIZE'],
    stratify     = y_full,
    random_state = CONFIG['SEED']
)

df_test = pd.read_csv(CONFIG['TEST_PATH'])
y_test  = df_test[CONFIG['TARGET_COL']].values.astype(np.int64)
X_test  = df_test.drop(columns=[CONFIG['TARGET_COL']]).values.astype(np.float32)

print(f"\nTraining pool : {len(X_train)} patients"
      f"  (PCOS=0: {(y_train==0).sum()}  PCOS=1: {(y_train==1).sum()})")
print(f"Test set      : {len(X_test)} patients"
      f"  (PCOS=0: {(y_test==0).sum()}   PCOS=1: {(y_test==1).sum()})")

assert len(X_test) == 109, f"Expected 109 test patients, got {len(X_test)}"
print("\nSanity check passed: test set = 109 patients.")

Full dataset: 541 rows x 48 features

Training pool : 432 patients  (PCOS=0: 291  PCOS=1: 141)
Test set      : 109 patients  (PCOS=0: 73   PCOS=1: 36)

Sanity check passed: test set = 109 patients.


## Cell 4 — Graph & Node2Vec Utility Functions

In [22]:
# CELL 4 — GRAPH AND NODE2VEC UTILITY FUNCTIONS

def build_knn_graph(features_scaled, k=10):
    n   = len(features_scaled)
    sim = cosine_similarity(features_scaled)
    np.fill_diagonal(sim, -2.0)
    src, dst, wts = [], [], []
    for i in range(n):
        top_k = np.argpartition(sim[i], -k)[-k:]
        for j in top_k:
            w = float(max(0.0, sim[i][j]))
            src += [i, j];  dst += [j, i];  wts += [w, w]
    ei = torch.tensor([src, dst], dtype=torch.long)
    ew = torch.tensor(wts,        dtype=torch.float)
    ei, ew = coalesce(ei, ew, num_nodes=n, reduce='max')
    return ei, ew


def add_synthetic_nodes(ei, ew, X_real_sc, X_syn_sc, k=10):
    n_real, n_syn = len(X_real_sc), len(X_syn_sc)
    if n_syn == 0:
        return ei, ew
    sim = cosine_similarity(X_syn_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(n_syn):
        s = n_real + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [s, j];  dst += [j, s];  wts += [w, w]
    new_ei = torch.tensor([src, dst], dtype=torch.long)
    new_ew = torch.tensor(wts,        dtype=torch.float)
    aug_ei = torch.cat([ei, new_ei], dim=1)
    aug_ew = torch.cat([ew, new_ew])
    aug_ei, aug_ew = coalesce(aug_ei, aug_ew,
                               num_nodes=n_real + n_syn, reduce='max')
    return aug_ei, aug_ew


def add_test_nodes(ei_aug, ew_aug, X_real_sc, X_test_sc,
                    k=10, n_train_aug=None):
    n_real = len(X_real_sc)
    if n_train_aug is None:
        n_train_aug = n_real
    sim = cosine_similarity(X_test_sc, X_real_sc)
    src, dst, wts = [], [], []
    for i in range(len(X_test_sc)):
        v = n_train_aug + i
        for j in np.argpartition(sim[i], -k)[-k:]:
            w = float(max(0.0, sim[i][j]))
            src += [v, j];  dst += [j, v];  wts += [w, w]
    new_ei  = torch.tensor([src, dst], dtype=torch.long)
    new_ew  = torch.tensor(wts,        dtype=torch.float)
    comb_ei = torch.cat([ei_aug, new_ei],  dim=1)
    comb_ew = torch.cat([ew_aug, new_ew])
    return comb_ei, comb_ew


def _random_walks(edge_index, num_nodes, walk_length, walks_per_node, seed=42):
    import random as _r
    _r.seed(seed)
    adj = [[] for _ in range(num_nodes)]
    ei  = edge_index.cpu().numpy()
    for s, d in zip(ei[0], ei[1]):
        adj[int(s)].append(int(d))
    walks = []
    nodes = list(range(num_nodes))
    for _ in range(walks_per_node):
        _r.shuffle(nodes)
        for start in nodes:
            walk = [start]
            for _ in range(walk_length - 1):
                nbrs = adj[walk[-1]]
                if nbrs:
                    walk.append(_r.choice(nbrs))
                else:
                    break
            walks.append([str(n) for n in walk])
    return walks


def train_node2vec(edge_index, num_nodes, cfg):
    walks = _random_walks(edge_index, num_nodes,
                           cfg['N2V_WALK_LEN'], cfg['N2V_WALKS'],
                           cfg['SEED'])
    w2v = Word2Vec(sentences=walks, vector_size=cfg['N2V_DIM'],
                    window=cfg['N2V_CONTEXT'], min_count=0,
                    sg=1, workers=1, seed=cfg['SEED'],
                    epochs=cfg['N2V_EPOCHS'])
    emb = np.zeros((num_nodes, cfg['N2V_DIM']), dtype=np.float32)
    for i in range(num_nodes):
        if str(i) in w2v.wv:
            emb[i] = w2v.wv[str(i)]
    return emb


def inductive_n2v(X_new_sc, X_train_sc, n2v_train, k=10):
    sim = cosine_similarity(X_new_sc, X_train_sc)
    out = np.zeros((len(X_new_sc), n2v_train.shape[1]), dtype=np.float32)
    for i in range(len(X_new_sc)):
        top_k = np.argpartition(sim[i], -k)[-k:]
        w     = np.maximum(sim[i][top_k], 0.0)
        wsum  = w.sum()
        w     = w / wsum if wsum > 1e-9 else np.ones(k) / k
        out[i] = (n2v_train[top_k] * w[:, None]).sum(axis=0)
    return out


def apply_smote(X, y, seed=42):
    smote        = SMOTE(random_state=seed, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X, y)
    n_syn = len(X_res) - len(X)
    print(f"  SMOTE: +{n_syn} synthetic PCOS+ "
          f"-> {(y_res==0).sum()} neg : {(y_res==1).sum()} pos")
    return X_res, y_res

print("Utility functions loaded.")

Utility functions loaded.


## Cell 5 — All Five Model Definitions

In [23]:
# CELL 5 — ALL FIVE MODEL DEFINITIONS

class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1 = GCNConv(in_ch, hidden_ch)
        self.c2 = GCNConv(hidden_ch, hidden_ch)
        self.lin = Linear(hidden_ch, out_ch)
        self.drop = dropout
    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index, edge_weight))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index, edge_weight))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class GAT(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, heads=4, dropout=0.3):
        super().__init__()
        self.c1 = GATConv(in_ch, hidden_ch,
                           heads=heads, dropout=dropout, concat=True)
        self.c2 = GATConv(hidden_ch * heads, hidden_ch,
                           heads=1, dropout=dropout, concat=False)
        self.lin = Linear(hidden_ch, out_ch)
        self.drop = dropout
    def forward(self, x, edge_index, edge_weight=None):
        x = F.dropout(x, self.drop, self.training)
        x = F.elu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.elu(self.c2(x, edge_index))
        return self.lin(x)


class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1 = SAGEConv(in_ch, hidden_ch)
        self.c2 = SAGEConv(hidden_ch, hidden_ch)
        self.lin = Linear(hidden_ch, out_ch)
        self.drop = dropout
    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class MPNN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1  = NNConv(in_ch, hidden_ch,
                           Linear(1, in_ch * hidden_ch), aggr='mean')
        self.c2  = NNConv(hidden_ch, hidden_ch,
                           Linear(1, hidden_ch * hidden_ch), aggr='mean')
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout
    def _ea(self, ew, ei, device):
        if ew is not None:
            return ew.unsqueeze(-1)
        return torch.ones(ei.size(1), 1, device=device)
    def forward(self, x, edge_index, edge_weight=None):
        ea = self._ea(edge_weight, edge_index, x.device)
        x  = F.relu(self.c1(x, edge_index, ea))
        x  = F.dropout(x, self.drop, self.training)
        x  = F.relu(self.c2(x, edge_index, ea))
        x  = F.dropout(x, self.drop, self.training)
        return self.lin(x)


class GIN(torch.nn.Module):
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.c1 = GINConv(Sequential(
            Linear(in_ch, hidden_ch), BatchNorm1d(hidden_ch),
            ReLU(), Linear(hidden_ch, hidden_ch)), train_eps=True)
        self.c2 = GINConv(Sequential(
            Linear(hidden_ch, hidden_ch), BatchNorm1d(hidden_ch),
            ReLU(), Linear(hidden_ch, hidden_ch)), train_eps=True)
        self.lin  = Linear(hidden_ch, out_ch)
        self.drop = dropout
    def forward(self, x, edge_index, edge_weight=None):
        x = F.relu(self.c1(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        x = F.relu(self.c2(x, edge_index))
        x = F.dropout(x, self.drop, self.training)
        return self.lin(x)


MODEL_REGISTRY = {
    'GCN'      : GCN,
    'GAT'      : GAT,
    'GRAPHSAGE': GraphSAGE,
    'MPNN'     : MPNN,
    'GIN'      : GIN,
}

def get_model(name, in_ch, hidden_ch, out_ch, dropout):
    name = name.upper()
    if name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{name}'")
    if name == 'GAT':
        # GAT has heads before dropout in its signature,
        # so dropout must be passed as a keyword argument
        return MODEL_REGISTRY[name](in_ch, hidden_ch, out_ch,
                                     dropout=dropout)
    return MODEL_REGISTRY[name.upper()](in_ch, hidden_ch, out_ch, dropout)

print("All five models defined: GCN | GAT | GraphSAGE | MPNN | GIN")

All five models defined: GCN | GAT | GraphSAGE | MPNN | GIN


## Cell 6 — Training, Evaluation & Metrics

In [24]:
# CELL 6 — TRAINING, EVALUATION AND METRICS

def train_one_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index, data.edge_weight)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    if torch.isnan(loss):
        return float('nan')
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_model(model, data, mask):
    model.eval()
    out   = model(data.x, data.edge_index, data.edge_weight)
    probs = torch.softmax(out[mask], dim=1)[:, 1].cpu().numpy()
    preds = out[mask].argmax(dim=1).cpu().numpy()
    true  = data.y[mask].cpu().numpy()
    return preds, probs, true


def compute_metrics(true, preds, probs):
    cm_arr       = confusion_matrix(true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm_arr.ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    try:    auroc = roc_auc_score(true, probs)
    except: auroc = 0.0
    return {
        'accuracy'    : accuracy_score(true, preds),
        'precision'   : precision_score(true, preds, zero_division=0),
        'recall'      : recall_score(true, preds, zero_division=0),
        'f1'          : f1_score(true, preds, zero_division=0),
        'mcc'         : matthews_corrcoef(true, preds),
        'auroc'       : auroc,
        'auprc'       : average_precision_score(true, probs),
        'sensitivity' : sens,
        'specificity' : spec,
        'kappa'       : cohen_kappa_score(true, preds),
        'fmi'         : fowlkes_mallows_score(true, preds),
        'nmi'         : normalized_mutual_info_score(true, preds),
    }, (int(tn), int(fp), int(fn), int(tp))

print("Training, evaluation and metric functions loaded.")

Training, evaluation and metric functions loaded.


## Cell 7 — All Visualisation Functions

In [25]:
# CELL 7 — ALL VISUALISATION FUNCTIONS

OUT = CONFIG['OUTPUT_DIR']

def plot_loss_curve(losses, model_name):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(1, len(losses)+1), losses, color='#3a86ff', lw=2)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
    ax.set_title(f'{model_name} Final Model — Training Loss Curve',
                  fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_loss.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_tsne(n2v_train, y_tr, n2v_test, y_te, model_name):
    all_emb    = np.vstack([n2v_train, n2v_test])
    all_labels = np.concatenate([y_tr, y_te])
    set_seed(GLOBAL_SEED)
    coords = TSNE(n_components=2, random_state=GLOBAL_SEED,
                   perplexity=min(30, len(all_emb)//4)
                   ).fit_transform(all_emb)
    n_tr = len(n2v_train)
    fig, ax = plt.subplots(figsize=(9, 7))
    for lbl, col, nm in [(0,'#3a86ff','No PCOS'),(1,'#ff595e','PCOS')]:
        mtr = (all_labels[:n_tr] == lbl)
        mte = (all_labels[n_tr:] == lbl)
        ax.scatter(coords[:n_tr][mtr,0], coords[:n_tr][mtr,1],
                    c=col, s=18, alpha=0.6, marker='o',
                    label=f'{nm} (train)', edgecolors='none')
        ax.scatter(coords[n_tr:][mte,0], coords[n_tr:][mte,1],
                    c=col, s=80, alpha=0.95, marker='*',
                    label=f'{nm} (test)', edgecolors='black', linewidths=0.4)
    ax.set_title(f'{model_name} — Node2Vec Embeddings (t-SNE)\n'
                  'Circles = Training  |  Stars = Test',
                  fontsize=11, fontweight='bold')
    ax.legend(fontsize=9, ncol=2)
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_tsne.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_confusion(true, preds, model_name):
    cm_arr = confusion_matrix(true, preds, labels=[0, 1])
    tn, fp, fn, tp = cm_arr.ravel()
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm_arr, annot=True, fmt='d', cmap='Blues', ax=ax,
                 xticklabels=['Pred: No PCOS', 'Pred: PCOS'],
                 yticklabels=['True: No PCOS', 'True: PCOS'],
                 linewidths=0.5)
    ax.set_title(f'{model_name} — Confusion Matrix (Test Set n=109)\n'
                  f'TP={tp}  FP={fp}  TN={tn}  FN={fn}',
                  fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_confusion.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_roc_pr(true, probs, model_name):
    fpr, tpr, _  = roc_curve(true, probs)
    pre, rec, _  = precision_recall_curve(true, probs)
    auroc_v = roc_auc_score(true, probs)
    auprc_v = average_precision_score(true, probs)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    ax1.plot(fpr, tpr, '#3a86ff', lw=2,
              label=f'ROC  (AUROC = {auroc_v:.4f})')
    ax1.plot([0,1],[0,1],'k--',lw=1)
    ax1.axhline(0.69, color='#06d6a0', ls=':', lw=1,
                 label='Supervisor threshold (0.69)')
    ax1.set_xlabel('False Positive Rate')
    ax1.set_ylabel('True Positive Rate')
    ax1.set_title(f'{model_name} — ROC Curve (Test Set)',
                   fontweight='bold')
    ax1.legend(); ax1.set_xlim(0,1); ax1.set_ylim(0,1.02)
    base = true.mean()
    ax2.plot(rec, pre, '#ff595e', lw=2,
              label=f'PR   (AUPRC = {auprc_v:.4f})')
    ax2.axhline(base, color='gray', ls='--', lw=1,
                 label=f'Baseline ({base:.3f})')
    ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
    ax2.set_title(f'{model_name} — Precision-Recall Curve (Test Set)',
                   fontweight='bold')
    ax2.legend(); ax2.set_xlim(0,1); ax2.set_ylim(0,1.02)
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_roc_pr.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_det(true, probs, model_name):
    fpr, fnr, _ = det_curve(true, probs)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(fpr, fnr, color='#8338ec', lw=2, label=model_name)
    ax.plot([0,1],[1,0],'k--',lw=1,alpha=0.5,label='Random')
    ax.scatter([0],[0],color='#06d6a0',s=80,zorder=5,label='Perfect (0,0)')
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('False Negative Rate (1 - Sensitivity)')
    ax.set_title(f'{model_name} — DET Curve (Test Set)',
                  fontweight='bold')
    ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_det.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_lift(true, probs, model_name):
    true  = np.array(true)
    probs = np.array(probs)
    n     = len(true)
    pos_rate = true.mean()
    order    = np.argsort(probs)[::-1]
    y_sorted = true[order]
    cum_pos  = np.cumsum(y_sorted)
    cum_pct  = np.arange(1, n+1) / n
    lift     = (cum_pos / np.arange(1, n+1)) / pos_rate
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(cum_pct*100, lift, color='#3a86ff', lw=2, label='Model lift')
    ax1.axhline(1.0, color='gray', ls='--', lw=1.5,
                 label='Baseline (random)')
    ax1.set_xlabel('% of Patients Selected')
    ax1.set_ylabel('Cumulative Lift')
    ax1.set_title(f'{model_name} — Lift Curve (Test Set)',
                   fontweight='bold')
    ax1.legend(); ax1.set_xlim(0,100); ax1.set_ylim(0)
    cum_gain = cum_pos / true.sum() * 100
    ax2.plot(cum_pct*100, cum_gain, color='#ff595e', lw=2,
              label='Model gains')
    ax2.plot([0,100],[0,100],'k--',lw=1,label='Random (baseline)')
    ax2.plot([0,pos_rate*100,100],[0,100,100],
              color='#06d6a0',ls=':',lw=1.5,label='Perfect model')
    ax2.set_xlabel('% of Patients Selected')
    ax2.set_ylabel('% of PCOS+ Cases Captured')
    ax2.set_title(f'{model_name} — Cumulative Gains (Test Set)',
                   fontweight='bold')
    ax2.legend(); ax2.set_xlim(0,100); ax2.set_ylim(0,105)
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_lift.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_precision_threshold(true, probs, model_name):
    prec, rec, thresholds = precision_recall_curve(true, probs)
    thresholds = np.append(thresholds, 1.0)
    f1_scores  = np.where((prec+rec)>0, 2*prec*rec/(prec+rec), 0)
    best_f1    = np.argmax(f1_scores)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(thresholds, prec, '#3a86ff', lw=2, label='Precision')
    ax.plot(thresholds, rec,  '#ff595e', lw=2, label='Recall')
    ax.plot(thresholds, f1_scores, '#06d6a0', lw=2, label='F1')
    ax.axvline(thresholds[best_f1], color='#ffbe0b', ls='--', lw=1.5,
                label=f'Best F1 threshold = {thresholds[best_f1]:.3f}')
    ax.axvline(0.5, color='gray', ls=':', lw=1, label='Default (0.5)')
    ax.set_xlabel('Classification Threshold')
    ax.set_ylabel('Score')
    ax.set_title(f'{model_name} — Precision / Recall / F1 vs Threshold',
                  fontweight='bold')
    ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    plt.tight_layout()
    plt.savefig(f'{OUT}{model_name}_final_prec_thresh.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


# ── comparison plots ──────────────────────────────────────────────

def plot_roc_overlay(all_results):
    colors = ['#3a86ff','#ff595e','#06d6a0','#ffbe0b','#8338ec']
    fig, ax = plt.subplots(figsize=(8, 7))
    for (m, res), color in zip(all_results.items(), colors):
        fpr, tpr, _ = roc_curve(res['true'], res['probs'])
        ax.plot(fpr, tpr, color=color, lw=2,
                 label=f"{m}  (AUROC = {res['metrics']['auroc']:.4f})")
    ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5,label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=11)
    ax.set_ylabel('True Positive Rate', fontsize=11)
    ax.set_title('ROC Curves — All Five Models\nFinal Test Set (n=109)',
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right')
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    plt.tight_layout()
    plt.savefig(f'{OUT}ALL_MODELS_roc_overlay.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_pr_overlay(all_results):
    colors = ['#3a86ff','#ff595e','#06d6a0','#ffbe0b','#8338ec']
    fig, ax = plt.subplots(figsize=(8, 7))
    for (m, res), color in zip(all_results.items(), colors):
        pre, rec, _ = precision_recall_curve(res['true'], res['probs'])
        ax.plot(rec, pre, color=color, lw=2,
                 label=f"{m}  (AUPRC = {res['metrics']['auprc']:.4f})")
    base = list(all_results.values())[0]['true'].mean()
    ax.axhline(base, color='gray', ls='--', lw=1,
                label=f'Baseline ({base:.3f})')
    ax.set_xlabel('Recall', fontsize=11)
    ax.set_ylabel('Precision', fontsize=11)
    ax.set_title('Precision-Recall Curves — All Five Models\nFinal Test Set (n=109)',
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower left')
    ax.set_xlim(0,1); ax.set_ylim(0,1.02)
    plt.tight_layout()
    plt.savefig(f'{OUT}ALL_MODELS_pr_overlay.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_metric_comparison(all_results):
    metric_keys = ['accuracy','precision','recall','f1','mcc',
                    'auroc','auprc','sensitivity','specificity',
                    'kappa','fmi','nmi']
    model_names = list(all_results.keys())
    colors = ['#3a86ff','#ff595e','#06d6a0','#ffbe0b','#8338ec']
    x, bw = np.arange(len(metric_keys)), 0.15
    fig, ax = plt.subplots(figsize=(18, 6))
    for i, (m, color) in enumerate(zip(model_names, colors)):
        vals = [all_results[m]['metrics'][k] for k in metric_keys]
        offset = (i - len(model_names)/2 + 0.5) * bw
        ax.bar(x + offset, vals, bw, label=m,
                color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([k.upper() for k in metric_keys],
                        rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Score'); ax.set_ylim(0, 1.08)
    ax.set_title('All Five Models — Final Test Set Metric Comparison',
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, ncol=5, loc='upper right')
    plt.tight_layout()
    plt.savefig(f'{OUT}ALL_MODELS_metric_comparison.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def plot_final_ranking(all_results):
    model_names = list(all_results.keys())
    mccs   = [all_results[m]['metrics']['mcc']   for m in model_names]
    aurocs = [all_results[m]['metrics']['auroc'] for m in model_names]
    order  = np.argsort(mccs)[::-1]
    sorted_names = [model_names[i] for i in order]
    sorted_mccs  = [mccs[i]   for i in order]
    sorted_auroc = [aurocs[i] for i in order]
    c_sorted = ['#06d6a0' if i==0 else '#3a86ff'
                 for i in range(len(sorted_names))]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    bars1 = ax1.bar(sorted_names, sorted_mccs, color=c_sorted,
                     alpha=0.85, edgecolor='white')
    for bar, v in zip(bars1, sorted_mccs):
        ax1.text(bar.get_x()+bar.get_width()/2, v+0.01, f'{v:.4f}',
                  ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax1.set_ylabel('MCC Score'); ax1.set_ylim(0, 1.0)
    ax1.set_title('Final Ranking by MCC\nGreen = Best', fontweight='bold')
    ax1.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.5)
    bars2 = ax2.bar(sorted_names, sorted_auroc, color=c_sorted,
                     alpha=0.85, edgecolor='white')
    for bar, v in zip(bars2, sorted_auroc):
        ax2.text(bar.get_x()+bar.get_width()/2, v+0.005, f'{v:.4f}',
                  ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.set_ylabel('AUROC'); ax2.set_ylim(0, 1.05)
    ax2.axhline(0.69, color='#ff595e', ls='--', lw=1.5,
                 label='Supervisor threshold (0.69)')
    ax2.set_title('Final Ranking by AUROC', fontweight='bold')
    ax2.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'{OUT}ALL_MODELS_final_ranking.png',
                dpi=150, bbox_inches='tight')
    plt.show(); plt.close()


def print_final_table(all_results):
    models = list(all_results.keys())
    met_list = ['accuracy','precision','recall','f1','mcc',
                 'auroc','auprc','sensitivity','specificity',
                 'kappa','fmi','nmi']
    W   = 12
    sep = '-' * (18 + W * len(models))
    print(f"\n{'='*len(sep)}")
    print(f"  FINAL TEST SET RESULTS — ALL FIVE MODELS  (n=109)")
    print(f"{'='*len(sep)}")
    print(f"{'Metric':<18}" + ''.join(f"{m:>{W}}" for m in models))
    print(sep)
    for met in met_list:
        vals = [all_results[m]['metrics'][met] for m in models]
        best = max(vals)
        row  = f"  {met.upper():<16}"
        for v in vals:
            star = '*' if abs(v-best) < 0.00001 else ' '
            row += f"{v:>{W-1}.4f}{star}"
        row += '  <- KEY METRIC' if met == 'mcc' else ''
        print(row)
    print(sep)
    print("  * = best value for each metric")
    print(f"\n  CONFUSION MATRIX SUMMARY")
    print(f"  {'Model':<12} {'TP':>6} {'FP':>6} {'TN':>6} {'FN':>6}")
    print(f"  {'-'*34}")
    for m in models:
        tn, fp, fn, tp = all_results[m]['cm']
        print(f"  {m:<12} {tp:>6} {fp:>6} {tn:>6} {fn:>6}")
    print(f"\n  PER-CLASS METRICS (Class 1 = PCOS positive)")
    print(f"  {'Model':<12} {'Cls1 P':>8} {'Cls1 R':>8} {'Cls1 F1':>9}")
    print(f"  {'-'*40}")
    for m in models:
        tn, fp, fn, tp = all_results[m]['cm']
        p1 = tp/(tp+fp) if (tp+fp)>0 else 0
        r1 = tp/(tp+fn) if (tp+fn)>0 else 0
        f1v= 2*p1*r1/(p1+r1) if (p1+r1)>0 else 0
        print(f"  {m:<12} {p1:>8.4f} {r1:>8.4f} {f1v:>9.4f}")
    print(f"{'='*len(sep)}")

print("All visualisation functions loaded.")

All visualisation functions loaded.


## Cell 8 — Data Preparation (built once, shared by all models)

In [26]:
# CELL 8 — DATA PREPARATION (built once, shared by all models)
# Graph, Node2Vec, SMOTE identical for all five architectures

print(f"\n{'#'*60}")
print("  DATA PREPARATION — built once, shared by all 5 models")
print(f"{'#'*60}\n")

set_seed(CONFIG['SEED'])
n_clinical = X_train.shape[1]

print("[1] Fitting StandardScaler...")
scaler  = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print("[2] Building k-NN training graph...", end=' ', flush=True)
ei_tr, ew_tr = build_knn_graph(X_tr_sc, k=CONFIG['K_NEIGHBOURS'])
n_real = len(X_tr_sc)
print(f"nodes={n_real}  edges={ei_tr.size(1)}")

print("[3] Training Node2Vec...", end=' ', flush=True)
n2v_tr = train_node2vec(ei_tr, n_real, CONFIG)
print(f"shape={n2v_tr.shape}")

X_tr_full = np.concatenate([X_tr_sc, n2v_tr], axis=1)

print("[4] Applying SMOTE...")
X_tr_sm, y_tr_sm = apply_smote(X_tr_full, y_train, CONFIG['SEED'])
n_syn       = len(X_tr_sm) - n_real
n_train_aug = len(X_tr_sm)

if n_syn > 0:
    X_syn_clin = X_tr_sm[n_real:, :n_clinical]
    ei_aug, ew_aug = add_synthetic_nodes(
        ei_tr, ew_tr, X_tr_sc, X_syn_clin,
        k=CONFIG['K_NEIGHBOURS'])
else:
    ei_aug, ew_aug = ei_tr, ew_tr

print("[5] Inductive Node2Vec for test patients...", end=' ', flush=True)
n2v_te    = inductive_n2v(X_te_sc, X_tr_sc, n2v_tr,
                           k=CONFIG['K_NEIGHBOURS'])
X_te_full = np.concatenate([X_te_sc, n2v_te], axis=1)
print("done")

ei_final, ew_final = add_test_nodes(
    ei_aug, ew_aug, X_tr_sc, X_te_sc,
    k=CONFIG['K_NEIGHBOURS'], n_train_aug=n_train_aug)

X_all = np.concatenate([X_tr_sm, X_te_full], axis=0)
y_all = np.concatenate([y_tr_sm, y_test])
n_all = len(X_all)

train_mask = torch.zeros(n_all, dtype=torch.bool)
train_mask[:n_train_aug] = True
test_mask  = torch.zeros(n_all, dtype=torch.bool)
test_mask[n_train_aug:]  = True

data = Data(
    x           = torch.tensor(X_all,   dtype=torch.float),
    edge_index  = ei_final,
    edge_weight = ew_final,
    y           = torch.tensor(y_all,   dtype=torch.long),
    train_mask  = train_mask,
    test_mask   = test_mask,
).to(DEVICE)

n_neg = float((y_tr_sm == 0).sum())
n_pos = float((y_tr_sm == 1).sum())
cw    = torch.tensor([1.0, n_neg / n_pos], dtype=torch.float).to(DEVICE)
in_channels = X_all.shape[1]   # 112

print(f"\n[6] PyG Data object ready.")
print(f"    Nodes: {n_all}  (train+synthetic={n_train_aug}, test=109)")
print(f"    Edges: {ei_final.size(1)}")
print(f"    Features per node: {data.x.shape[1]}")
print(f"    Class weights: PCOS=0 -> 1.000  |  PCOS=1 -> {n_neg/n_pos:.3f}")
print(f"\nAll 5 models will train on this identical data representation.")


############################################################
  DATA PREPARATION — built once, shared by all 5 models
############################################################

[1] Fitting StandardScaler...
[2] Building k-NN training graph... nodes=432  edges=5512
[3] Training Node2Vec... shape=(432, 64)
[4] Applying SMOTE...
  SMOTE: +150 synthetic PCOS+ -> 291 neg : 291 pos
[5] Inductive Node2Vec for test patients... done

[6] PyG Data object ready.
    Nodes: 691  (train+synthetic=582, test=109)
    Edges: 10692
    Features per node: 112
    Class weights: PCOS=0 -> 1.000  |  PCOS=1 -> 1.000

All 5 models will train on this identical data representation.


## Cell 9 — Train & Evaluate All Five Models

In [27]:
# CELL 9 — TRAIN AND EVALUATE ALL FIVE MODELS
# Each model trains on the same graph and data object
# Evaluated exactly once on the sealed test set

all_results = {}
all_losses  = {}

for model_name in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f"  TRAINING: {model_name}")
    print(f"  {CONFIG['FINAL_EPOCHS']} epochs  |  No early stopping  |  {DEVICE}")
    print(f"{'='*60}")

    set_seed(CONFIG['SEED'])
    model = get_model(model_name, in_channels,
                       CONFIG['HIDDEN_DIM'], 2,
                       CONFIG['DROPOUT']).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=CONFIG['LR'],
                                  weight_decay=CONFIG['WEIGHT_DECAY'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=20, min_lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss(weight=cw)

    n_params = sum(p.numel() for p in model.parameters()
                   if p.requires_grad)
    print(f"  Parameters: {n_params:,}")

    losses = []
    for epoch in range(1, CONFIG['FINAL_EPOCHS'] + 1):
        loss = train_one_epoch(model, data, optimizer, criterion)
        if loss != loss:
            print(f"  NaN loss at epoch {epoch}. Stopping.")
            break
        losses.append(loss)
        scheduler.step(loss)
        if epoch % 30 == 0:
            print(f"  Epoch {epoch:3d}/{CONFIG['FINAL_EPOCHS']}  "
                  f"loss={loss:.4f}  "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    print(f"  Training done. Final loss: {losses[-1]:.4f}")
    all_losses[model_name] = losses

    # Evaluate on sealed test set
    preds, probs, true = evaluate_model(model, data, data.test_mask)
    metrics, cm_vals   = compute_metrics(true, preds, probs)
    all_results[model_name] = {
        'metrics': metrics, 'cm': cm_vals,
        'true': true, 'probs': probs, 'preds': preds,
    }

    tn, fp, fn, tp = cm_vals
    n0 = int((true==0).sum()); n1 = int((true==1).sum())
    p1 = tp/(tp+fp) if (tp+fp)>0 else 0
    r1 = tp/(tp+fn) if (tp+fn)>0 else 0
    f1v= 2*p1*r1/(p1+r1) if (p1+r1)>0 else 0

    print(f"\n  TEST SET RESULTS (n=109):")
    for k, v in metrics.items():
        flag = '  <- KEY METRIC'    if k=='mcc'                       else (
               '  <- THRESHOLD MET' if k=='auroc' and v>=0.69         else (
               '  <- BELOW THRESHOLD' if k=='auroc'                   else ''))
        print(f"    {k.upper():<16} {v:.4f}{flag}")
    print(f"  TP={tp}  FP={fp}  TN={tn}  FN={fn}")
    print(f"  Class 1 (PCOS): P={p1:.4f}  R={r1:.4f}  F1={f1v:.4f}")

    # Per-model plots
    plot_loss_curve(losses, model_name)
    plot_tsne(n2v_tr, y_train, n2v_te, y_test, model_name)
    plot_confusion(true, preds, model_name)
    plot_roc_pr(true, probs, model_name)
    plot_det(true, probs, model_name)
    plot_lift(true, probs, model_name)
    plot_precision_threshold(true, probs, model_name)

    torch.save(model.state_dict(),
                f'{OUT}{model_name}_final_weights.pt')
    print(f"  Weights saved -> {model_name}_final_weights.pt")

    del model, optimizer, scheduler, criterion
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


  TRAINING: GCN
  150 epochs  |  No early stopping  |  cuda
  Parameters: 11,522
  Epoch  30/150  loss=0.3265  lr=1.00e-03
  Epoch  60/150  loss=0.2555  lr=1.00e-03
  Epoch  90/150  loss=0.2206  lr=1.00e-03
  Epoch 120/150  loss=0.1925  lr=1.00e-03
  Epoch 150/150  loss=0.1857  lr=1.00e-03
  Training done. Final loss: 0.1857

  TEST SET RESULTS (n=109):
    ACCURACY         0.8532
    PRECISION        0.7941
    RECALL           0.7500
    F1               0.7714
    MCC              0.6640  <- KEY METRIC
    AUROC            0.9091  <- THRESHOLD MET
    AUPRC            0.8716
    SENSITIVITY      0.7500
    SPECIFICITY      0.9041
    KAPPA            0.6635
    FMI              0.7744
    NMI              0.3559
  TP=27  FP=7  TN=66  FN=9
  Class 1 (PCOS): P=0.7941  R=0.7500  F1=0.7714
  Weights saved -> GCN_final_weights.pt

  TRAINING: GAT
  150 epochs  |  No early stopping  |  cuda
  Parameters: 46,146
  Epoch  30/150  loss=0.2651  lr=1.00e-03
  Epoch  60/150  loss=0.2366  lr=1.

## Cell 10 — Comparison Plots & Final Summary

In [28]:
# CELL 10 — COMPARISON PLOTS AND FINAL SUMMARY

print(f"\n{'#'*60}")
print("  GENERATING COMPARISON PLOTS")
print(f"{'#'*60}")

plot_roc_overlay(all_results)
plot_pr_overlay(all_results)
plot_metric_comparison(all_results)
plot_final_ranking(all_results)

print_final_table(all_results)

# Save results CSV
rows = []
for m_name, res in all_results.items():
    row = {'Model': m_name}
    row.update(res['metrics'])
    tn, fp, fn, tp = res['cm']
    row.update({'TP':tp,'FP':fp,'TN':tn,'FN':fn})
    rows.append(row)

pd.DataFrame(rows).set_index('Model').to_csv(
    f'{OUT}ALL_MODELS_final_test_results.csv')
print(f"\nResults CSV saved -> ALL_MODELS_final_test_results.csv")

print(f"\n{'='*52}")
print("  FINAL RANKING BY MCC — TEST SET")
print(f"{'='*52}")
ranked = sorted(all_results.items(),
                key=lambda x: x[1]['metrics']['mcc'], reverse=True)
for i, (m, res) in enumerate(ranked):
    mcc   = res['metrics']['mcc']
    auroc = res['metrics']['auroc']
    acc   = res['metrics']['accuracy']
    f1    = res['metrics']['f1']
    flag  = 'PASS' if auroc >= 0.69 else 'FAIL'
    print(f"  {i+1}. {m:<12}  MCC={mcc:.4f}  "
          f"AUROC={auroc:.4f}[{flag}]  "
          f"Acc={acc:.4f}  F1={f1:.4f}")
print(f"{'='*52}")


############################################################
  GENERATING COMPARISON PLOTS
############################################################

  FINAL TEST SET RESULTS — ALL FIVE MODELS  (n=109)
Metric                     GCN         GAT   GRAPHSAGE        MPNN         GIN
------------------------------------------------------------------------------
  ACCURACY             0.8532      0.8624      0.9174*     0.8349      0.7890 
  PRECISION            0.7941      0.7838      0.9091*     0.7812      0.6327 
  RECALL               0.7500      0.8056      0.8333      0.6944      0.8611*
  F1                   0.7714      0.7945      0.8696*     0.7353      0.7294 
  MCC                  0.6640      0.6912      0.8110*     0.6182      0.5810   <- KEY METRIC
  AUROC                0.9091      0.9231      0.9486*     0.9277      0.8904 
  AUPRC                0.8716      0.8869      0.9309*     0.8904      0.8591 
  SENSITIVITY          0.7500      0.8056      0.8333      0.6944   

In [30]:
import zipfile, os

output_dir  = '/kaggle/working/'
zip_path    = '/kaggle/working/PCOS_NB3_Final_Results.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in os.listdir(output_dir):
        if not file.endswith('.zip'):
            file_path = os.path.join(output_dir, file)
            if os.path.isfile(file_path):
                zipf.write(file_path, file)
                print(f"  Added: {file}")

print(f"\nZip created: {zip_path}")
print(f"Total files: {len([f for f in os.listdir(output_dir) if not f.endswith('.zip')])}")

zip_path

  Added: ALL_MODELS_final_test_results.csv
  Added: GIN_final_roc_pr.png
  Added: GCN_final_loss.png
  Added: GCN_final_confusion.png
  Added: GAT_final_weights.pt
  Added: GIN_final_det.png
  Added: GCN_final_lift.png
  Added: GCN_final_weights.pt
  Added: GAT_final_roc_pr.png
  Added: GRAPHSAGE_final_loss.png
  Added: GIN_final_tsne.png
  Added: GAT_final_tsne.png
  Added: MPNN_final_lift.png
  Added: MPNN_final_prec_thresh.png
  Added: MPNN_final_roc_pr.png
  Added: GAT_final_confusion.png
  Added: GIN_final_confusion.png
  Added: GRAPHSAGE_final_prec_thresh.png
  Added: MPNN_final_det.png
  Added: GAT_final_prec_thresh.png
  Added: GRAPHSAGE_final_confusion.png
  Added: GIN_final_weights.pt
  Added: GRAPHSAGE_final_lift.png
  Added: MPNN_final_loss.png
  Added: ALL_MODELS_metric_comparison.png
  Added: GAT_final_det.png
  Added: ALL_MODELS_roc_overlay.png
  Added: GIN_final_lift.png
  Added: GRAPHSAGE_final_roc_pr.png
  Added: GRAPHSAGE_final_weights.pt
  Added: MPNN_final_confusio

'/kaggle/working/PCOS_NB3_Final_Results.zip'